# Week 5 Exercise - Practical RAG Assistant (AdnanGobeljic)

This contribution follows the Week 5 flow:
- Day 1: baseline knowledge worker idea
- Day 2: chunking and embeddings
- Day 3: retriever + LLM answer generation
- Day 4: lightweight retrieval inspection
- Day 5: cleaner, reusable helper functions

The notebook builds a simple RAG assistant over the existing `week5/knowledge-base` content.

In [10]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv(override=True)
MODEL = "gpt-4.1-nano"
DB_NAME = "adnan_week5_vector_db"

In [ ]:
def resolve_week5_paths() -> tuple[Path, Path]:
    """Find week5 root regardless of where notebook is launched from."""
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]

    for base in candidates:
        week5_dir = base / "week5"
        if (week5_dir / "knowledge-base").exists():
            return week5_dir, week5_dir / "knowledge-base"

    raise FileNotFoundError("Could not locate week5/knowledge-base from current working directory")


week5_dir, kb_path = resolve_week5_paths()
db_path = week5_dir / "community-contributions" / "AdnanGobeljic" / DB_NAME
print(f"Week 5 root: {week5_dir}")
print(f"Knowledge base: {kb_path}")
print(f"Vector DB path: {db_path}")

In [ ]:
def load_and_chunk_documents(knowledge_base_path: Path, chunk_size: int = 700, chunk_overlap: int = 120):
    loader = DirectoryLoader(
        str(knowledge_base_path),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(documents)
    print(f"Loaded {len(documents)} docs -> {len(chunks)} chunks")
    return chunks


chunks = load_and_chunk_documents(kb_path)

In [ ]:
def build_vectorstore(chunks, persist_directory: Path, use_persistence: bool = False):
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    if use_persistence:
        persist_directory = persist_directory.resolve()
        persist_directory.parent.mkdir(parents=True, exist_ok=True)
        persist_directory.mkdir(parents=True, exist_ok=True)

        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory=str(persist_directory),
            collection_name="adnan_week5_docs",
        )
        print(f"Vector store ready with {vectorstore._collection.count()} vectors (persisted)")
        return vectorstore

    # In-memory mode avoids local sqlite file issues and is enough for this notebook flow.
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="adnan_week5_docs",
    )
    print(f"Vector store ready with {vectorstore._collection.count()} vectors (in-memory)")
    return vectorstore


vectorstore = build_vectorstore(chunks, db_path, use_persistence=False)
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10, "fetch_k": 40},
)

In [14]:
import re

llm = ChatOpenAI(model_name=MODEL, temperature=0)

SYSTEM_PROMPT = """
You are a helpful assistant for Insurellm employees.
Answer using only the retrieved context.
If the context is insufficient, say you do not have enough information.
Keep answers concise and factual.

Context:
{context}
"""


def _extract_query_terms(question: str):
    stopwords = {
        "what", "which", "who", "tell", "about", "the", "is", "in", "on", "of",
        "to", "can", "you", "me", "a", "an", "for", "and", "with", "from",
    }
    lower_terms = re.findall(r"[a-z0-9\-]+", question.lower())
    terms = [t for t in lower_terms if len(t) >= 4 and t not in stopwords]

    # Keep short all-caps acronyms like IIOTY as high-signal terms.
    acronyms = [a.lower() for a in re.findall(r"\b[A-Z]{2,}\b", question)]
    for acronym in acronyms:
        if acronym not in terms:
            terms.append(acronym)

    # Preserve order while removing duplicates.
    deduped = []
    seen = set()
    for t in terms:
        if t not in seen:
            deduped.append(t)
            seen.add(t)
    return deduped


def retrieve_context_docs(question: str, final_k: int = 8, keyword_pool: int = 40):
    semantic_docs = retriever.invoke(question)
    query_terms = _extract_query_terms(question)
    acronyms = [a.lower() for a in re.findall(r"\b[A-Z]{2,}\b", question)]
    years = re.findall(r"\b(?:19|20)\d{2}\b", question)

    term_weights = {term: 1 for term in query_terms}
    for acronym in acronyms:
        term_weights[acronym] = max(term_weights.get(acronym, 0), 8)
    for year in years:
        term_weights[year] = max(term_weights.get(year, 0), 4)

    keyword_matches = []
    for doc in chunks:
        text = doc.page_content.lower()
        weighted_score = sum(weight for term, weight in term_weights.items() if term in text)
        if weighted_score > 0:
            keyword_matches.append((weighted_score, doc))

    keyword_matches.sort(key=lambda item: item[0], reverse=True)
    keyword_docs = [doc for _, doc in keyword_matches[:keyword_pool]]

    candidates = []
    seen = set()
    keyword_score_by_key = {}

    for score, doc in keyword_matches[:keyword_pool]:
        key = (doc.metadata.get("source", ""), doc.page_content[:180])
        keyword_score_by_key[key] = max(keyword_score_by_key.get(key, 0), score)

    for doc in (semantic_docs + keyword_docs):
        key = (doc.metadata.get("source", ""), doc.page_content[:180])
        if key in seen:
            continue
        seen.add(key)
        candidates.append(doc)

    def score_doc(doc):
        text = doc.page_content.lower()
        key = (doc.metadata.get("source", ""), doc.page_content[:180])
        keyword_bonus = keyword_score_by_key.get(key, 0)

        semantic_bonus = 0
        if doc in semantic_docs:
            semantic_bonus = max(0, 10 - semantic_docs.index(doc))

        acronym_hit = any(acronym in text for acronym in acronyms)
        year_hit = any(year in text for year in years)
        direct_signal_bonus = 20 if (acronym_hit and year_hit) else 0

        return (3 * keyword_bonus) + semantic_bonus + direct_signal_bonus

    ranked = sorted(candidates, key=score_doc, reverse=True)
    return ranked[:final_k]


def answer_question(question: str) -> str:
    docs = retrieve_context_docs(question, final_k=8, keyword_pool=25)
    context = "\n\n".join(doc.page_content for doc in docs)
    prompt = SYSTEM_PROMPT.format(context=context)
    response = llm.invoke([
        SystemMessage(content=prompt),
        HumanMessage(content=question),
    ])
    return response.content


def inspect_retrieval(question: str, top_n: int = 3):
    docs = retrieve_context_docs(question, final_k=max(top_n, 8), keyword_pool=25)
    for idx, doc in enumerate(docs[:top_n], start=1):
        source = doc.metadata.get("source", "unknown")
        print(f"[{idx}] {source}")
        print(doc.page_content[:220].replace("\n", " "))
        print("-" * 80)

In [16]:
sample_questions = [
    "Who won IIOTY in 2023?",
    "Which product is focused on enterprise reinsurance?",
    "What can you tell me about Avery Lancaster?",
]

for q in sample_questions:
    print(f"\nQ: {q}")
    inspect_retrieval(q, top_n=2)
    print("A:", answer_question(q))


Q: Who won IIOTY in 2023?
[1] /Users/adnangobeljic/Programiranje/Andela/llm_engineering/week5/knowledge-base/employees/Oliver Spencer.md
- **2022**: **5/5** - Exceptional performance during the "Innovate" initiative, showcasing leadership and creativity. - **2023**: **3/5** - Maintaining steady work; expectations for innovation not fully met, leading to d
--------------------------------------------------------------------------------
[2] /Users/adnangobeljic/Programiranje/Andela/llm_engineering/week5/knowledge-base/employees/Oliver Spencer.md
## Annual Performance History - **2018**: **3/5** - Adaptable team player but still learning to take initiative. - **2019**: **4/5** - Demonstrated strong problem-solving skills, outstanding contribution on the claims pr
--------------------------------------------------------------------------------
A: I do not have enough information to answer that question.

Q: Which product is focused on enterprise reinsurance?
[1] /Users/adnangobeljic/Progr